# Análise das bases sugeridas

## Objetivo
Analisar qual base usar

## 📊 Bases de Dados Disponíveis (4 Bases)

Este notebook baixa e processa TODAS as 4 bases automaticamente:

| # | Base | Autor | Descrição | Registros |
|---|------|-------|-----------|----------|
| 1 | **bank-marketing** | henriqueyamahata | Campanhas bancárias, propensão de conversão | ~41k |
| 2 | **bank-marketing-data-set** | tunguz | Variação para comparação | ~45k |
| 3 | **bank-term-deposit-subscription** | dharmik34 | Assinatura de depósito a prazo | ~11k |
| 4 | **telemarketing-jyb-dataset** | aguado | Campanhas de contato e resposta | ~4k |

**Sistema de Cache:** Se os dados já existem, não baixa novamente! 🚀

---

## 1️⃣ Setup e Imports

In [37]:
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import joblib
import warnings
import os
import subprocess
import sys
from pathlib import Path

field_limit = sys.maxsize
while True:
    try:
        csv.field_size_limit(field_limit)
        break
    except OverflowError:
        field_limit //= 10

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Imports realizados com sucesso!')

Imports realizados com sucesso!


## 2️⃣ Configuração de Datasets e Cache

In [33]:
# Configuração das tabelas disponíveis para comparação
DATASETS_CONFIG = {
    'bank-marketing': {
        'name': '1️⃣ Bank Marketing (henriqueyamahata)',
        'kaggle_id': 'henriqueyamahata/bank-marketing',
        'description': 'Campanhas bancárias, propensão de conversão e decisão de oferta',
        'separator': ';',
        'target_col': 'y',
        'size_mb': '~0.4 MB'
    },
    'bank-marketing-data-set': {
        'name': '2️⃣ Bank Marketing Data Set (tunguz)',
        'kaggle_id': 'tunguz/bank-marketing-data-set',
        'description': 'Variação do problema de marketing bancário para comparação',
        'separator': ';',
        'target_col': 'y',
        'size_mb': '~1.0 MB'
    },
    'bank-term-deposit-subscription': {
        'name': '3️⃣ Bank Term Deposit Subscription (dharmik34)',
        'kaggle_id': 'dharmik34/bank-term-deposit-subscription',
        'description': 'Assinatura de depósito a prazo como proxy de conversão',
        'separator': ',',
        'target_col': 'target',
        'size_mb': '~0.2 MB'
    },
    'telemarketing-jyb-dataset': {
        'name': '4️⃣ Telemarketing JYB Dataset (aguado)',
        'kaggle_id': 'aguado/telemarketing-jyb-dataset',
        'description': 'Campanhas de contato e resposta, útil para comparação de canal',
        'separator': ',',
        'target_col': 'y',
        'size_mb': '~0.1 MB'
    }
}

RAW_DATA_DIR = Path('data/raw')
PROCESSED_DATA_DIR = Path('data/processed')
MODELS_DIR = Path('models')

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print('Configuração de datasets inicializada')
print(f'Raw: {RAW_DATA_DIR}')
print(f'Processed: {PROCESSED_DATA_DIR}')
print(f'Models: {MODELS_DIR}')

Configuração de datasets inicializada
Raw: data\raw
Processed: data\processed
Models: models


## 3️⃣ Sistema de Cache - Download Automático com Detecção

In [34]:
def dataset_exists(dataset_name):
    """Verifica se dataset já foi baixado (cache)"""
    dataset_dir = RAW_DATA_DIR / dataset_name
    if not dataset_dir.exists():
        return False
    
    # Verificar se tem arquivos CSV
    csv_files = list(dataset_dir.glob('*.csv'))
    return len(csv_files) > 0

def download_dataset(dataset_name, config):
    """Baixa dataset do Kaggle com cache"""
    dataset_dir = RAW_DATA_DIR / dataset_name
    
    # Verificar cache
    if dataset_exists(dataset_name):
        print(f"✅ {config['name']}")
        print(f"   📦 Cache encontrado - usando dados locais")
        csv_files = list(dataset_dir.glob('*.csv'))
        print(f"   📁 Arquivos: {[f.name for f in csv_files]}")
        return True
    
    # Baixar
    print(f"⬇️  {config['name']}")
    print(f"   📥 Baixando do Kaggle...")
    
    try:
        dataset_dir.mkdir(parents=True, exist_ok=True)
        
        # Baixar dataset
        subprocess.run([
            'kaggle', 'datasets', 'download',
            '-d', config['kaggle_id'],
            '-p', str(dataset_dir)
        ], check=True, capture_output=True)
        
        # Descompactar
        zip_files = list(dataset_dir.glob('*.zip'))
        for zip_file in zip_files:
            subprocess.run(f'cd {dataset_dir} && unzip -q {zip_file.name} && rm {zip_file.name}', 
                           shell=True, check=True)
        
        print(f"   ✅ Download concluído")
        csv_files = list(dataset_dir.glob('*.csv'))
        print(f"   📁 Arquivos: {[f.name for f in csv_files]}")
        return True
    
    except Exception as e:
        print(f"   ❌ Erro: {e}")
        print(f"   💡 Baixe manualmente: https://www.kaggle.com/datasets/{config['kaggle_id']}")
        return False

# Resumo das bases
print("\n" + "="*70)
print("📊 SISTEMA DE CACHE - VERIFICANDO DATASETS")
print("="*70)

for dataset_name, config in DATASETS_CONFIG.items():
    if dataset_exists(dataset_name):
        print(f"\n✅ {config['name']} [{config['size_mb']}]")
        print(f"   🎯 Cache encontrado")
    else:
        print(f"\n⬇️  {config['name']} [{config['size_mb']}]")
        print(f"   ⏳ Fila de download")

print("\n" + "="*70)


📊 SISTEMA DE CACHE - VERIFICANDO DATASETS

✅ 1️⃣ Bank Marketing (henriqueyamahata) [~0.4 MB]
   🎯 Cache encontrado

✅ 2️⃣ Bank Marketing Data Set (tunguz) [~1.0 MB]
   🎯 Cache encontrado

✅ 3️⃣ Bank Term Deposit Subscription (dharmik34) [~0.2 MB]
   🎯 Cache encontrado

✅ 4️⃣ Telemarketing JYB Dataset (aguado) [~0.1 MB]
   🎯 Cache encontrado



## 4️⃣ Executar Downloads com Cache

In [31]:
# Baixar todas as 4 bases (usando cache se existir)
print("\n" + "="*70)
print("📥 INICIANDO DOWNLOADS (COM CACHE)")
print("="*70 + "\n")

download_status = {}
for dataset_name, config in DATASETS_CONFIG.items():
    success = download_dataset(dataset_name, config)
    download_status[dataset_name] = success
    print()

# Resumo
print("="*70)
print("📊 RESUMO DE DOWNLOADS")
print("="*70)
success_count = sum(download_status.values())
total_count = len(download_status)
print(f"\n✅ {success_count}/{total_count} bases prontas")

if success_count == total_count:
    print("\n🎉 Todas as 4 bases foram baixadas com sucesso!")
else:
    print(f"\n⚠️  {total_count - success_count} base(s) falharam. Baixe manualmente.")


📥 INICIANDO DOWNLOADS (COM CACHE)

✅ 1️⃣ Bank Marketing (henriqueyamahata)
   📦 Cache encontrado - usando dados locais
   📁 Arquivos: ['bank-additional-full.csv']

✅ 2️⃣ Bank Marketing Data Set (tunguz)
   📦 Cache encontrado - usando dados locais
   📁 Arquivos: ['bank-additional-full.csv']

✅ 3️⃣ Bank Term Deposit Subscription (dharmik34)
   📦 Cache encontrado - usando dados locais
   📁 Arquivos: ['bank-full.csv']

✅ 4️⃣ Telemarketing JYB Dataset (aguado)
   📦 Cache encontrado - usando dados locais
   📁 Arquivos: ['test.csv', 'train.csv']

📊 RESUMO DE DOWNLOADS

✅ 4/4 bases prontas

🎉 Todas as 4 bases foram baixadas com sucesso!


## 5️⃣ Carregar Todos os Datasets

In [38]:
def load_dataset(dataset_name, config):
    """Carrega todos os CSVs e autodetecta o separador de cada arquivo."""
    dataset_dir = RAW_DATA_DIR / dataset_name
    csv_files = sorted(dataset_dir.glob('*.csv'))

    if not csv_files:
        print(f'Nenhum arquivo CSV encontrado em {dataset_dir}')
        return {}

    loaded_tables = {}
    for csv_file in csv_files:
        try:
            dataset_df = pd.read_csv(str(csv_file), sep=None, engine='python')
            loaded_tables[csv_file.stem] = dataset_df
        except Exception as error:
            print(f'Erro ao carregar {csv_file.name}: {error}')
    return loaded_tables

print('\n' + '=' * 70)
print('CARREGANDO TODAS AS TABELAS CSV')
print('=' * 70 + '\n')

datasets = {}
dataset_info_list = []

for dataset_name, config in DATASETS_CONFIG.items():
    loaded_tables = load_dataset(dataset_name, config)
    for table_name, dataset_df in loaded_tables.items():
        table_id = f'{dataset_name}/{table_name}'
        datasets[table_id] = dataset_df
        target_col = config['target_col']
        if target_col not in dataset_df.columns:
            for candidate in ['y', 'target', 'label', 'response']:
                if candidate in dataset_df.columns:
                    target_col = candidate
                    break
        dataset_info_list.append({
            'Dataset': table_id,
            'Registros': f'{dataset_df.shape[0]:,}',
            'Colunas': dataset_df.shape[1],
            'Target': target_col,
            'Memoria': f'{dataset_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB'
        })
        print(f'OK {table_id}')
        print(f'   Shape: {dataset_df.shape[0]:,} x {dataset_df.shape[1]}')
        print(f'   Colunas: {list(dataset_df.columns)}')
        print(f'   Target detectado: {target_col}')

print('=' * 70)
print('COMPARACAO DE TABELAS')
print('=' * 70)
df_info = pd.DataFrame(dataset_info_list)
print(df_info.to_string(index=False))
print(f'\n{len(datasets)} tabelas carregadas')


CARREGANDO TODAS AS TABELAS CSV

OK bank-marketing/bank-additional-full
   Shape: 41,188 x 21
   Colunas: ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']
   Target detectado: y
OK bank-marketing-data-set/bank-additional-full
   Shape: 41,188 x 21
   Colunas: ['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']
   Target detectado: y
OK bank-term-deposit-subscription/bank-full
   Shape: 45,211 x 17
   Colunas: ['age', 'job', 'marital', 'education', 'default', 'balance', 'housing', 'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'y']
   Target detectado: y
OK te

In [39]:
print('\n' + '=' * 80)
print('DISTRIBUIÇÃO DOS TARGETS POR TABELA')
print('=' * 80)

target_summary = []
for table_id, table_df in datasets.items():
    dataset_name = table_id.split('/', 1)[0]
    target_col = DATASETS_CONFIG[dataset_name]['target_col']
    if target_col not in table_df.columns:
        target_col = next(
            (candidate for candidate in ['y', 'target', 'label', 'response'] if candidate in table_df.columns),
            None
        )
    if target_col is None:
        print(f'\n{table_id}: nenhum target reconhecido')
        continue

    counts = table_df[target_col].value_counts(dropna=False)
    percentages = table_df[target_col].value_counts(normalize=True, dropna=False) * 100
    print(f'\n{table_id} | target: {target_col} | linhas: {len(table_df):,}')
    print(pd.DataFrame({
        'quantidade': counts,
        'percentual': percentages.round(2)
    }).to_string())

    if len(percentages) == 2:
        target_summary.append({
            'tabela': table_id,
            'target': target_col,
            'classe_majoritaria_%': round(float(percentages.max()), 2),
            'classe_minoritaria_%': round(float(percentages.min()), 2),
            'razao_min_max': round(float(percentages.min() / percentages.max()), 4)
        })

print('\n' + '=' * 80)
print('COMPARAÇÃO DE EQUILÍBRIO')
print('=' * 80)
if target_summary:
    balance_summary = pd.DataFrame(target_summary).sort_values('razao_min_max', ascending=False)
    print(balance_summary.to_string(index=False))
    print(f"\nMais equilibrada: {balance_summary.iloc[0]['tabela']}")
else:
    print('Não foi possível comparar tabelas com exatamente duas classes.')


DISTRIBUIÇÃO DOS TARGETS POR TABELA

bank-marketing/bank-additional-full | target: y | linhas: 41,188
     quantidade  percentual
y                          
no        36548       88.73
yes        4640       11.27

bank-marketing-data-set/bank-additional-full | target: y | linhas: 41,188
     quantidade  percentual
y                          
no        36548       88.73
yes        4640       11.27

bank-term-deposit-subscription/bank-full | target: y | linhas: 45,211
     quantidade  percentual
y                          
no        39922        88.3
yes        5289        11.7

telemarketing-jyb-dataset/test: nenhum target reconhecido

telemarketing-jyb-dataset/train | target: y | linhas: 28,645
     quantidade  percentual
y                          
no        25362       88.54
yes        3283       11.46

COMPARAÇÃO DE EQUILÍBRIO
                                      tabela target  classe_majoritaria_%  classe_minoritaria_%  razao_min_max
    bank-term-deposit-subscription/bank-full 

In [40]:
print('\n' + '=' * 90)
print('ANÁLISE DA COLUNA ARM E CONVERSÃO POR OPÇÃO')
print('=' * 90)

arm_candidates = [
    'contact', 'channel', 'arm', 'offer', 'offer_type',
    'treatment', 'action', 'campaign', 'canal'
]
arm_summary = []

for table_id, table_df in datasets.items():
    dataset_name = table_id.split('/', 1)[0]
    target_col = DATASETS_CONFIG[dataset_name]['target_col']
    if target_col not in table_df.columns:
        target_col = next(
            (candidate for candidate in ['y', 'target', 'label', 'response'] if candidate in table_df.columns),
            None
        )

    arm_col = next((column for column in arm_candidates if column in table_df.columns), None)
    if arm_col is None or target_col is None:
        print(f'\n{table_id}: coluna arm ou target não identificada')
        print(f'Colunas disponíveis: {list(table_df.columns)}')
        continue

    arm_data = table_df[[arm_col, target_col]].dropna().copy()
    positive_values = {'yes', 'y', 'sim', '1', 'true', 'converted', 'success'}
    target_as_text = arm_data[target_col].astype(str).str.strip().str.lower()
    arm_data['conversion'] = target_as_text.isin(positive_values).astype(int)

    by_arm = (
        arm_data.groupby(arm_col, dropna=False)
        .agg(
            observacoes=('conversion', 'size'),
            conversoes=('conversion', 'sum'),
            taxa_conversao=('conversion', 'mean')
        )
        .sort_values('taxa_conversao', ascending=False)
    )
    by_arm['taxa_conversao_%'] = (by_arm['taxa_conversao'] * 100).round(2)

    print(f'\n{table_id} | arm: {arm_col} | target: {target_col}')
    print(by_arm[['observacoes', 'conversoes', 'taxa_conversao_%']].to_string())

    if len(by_arm) > 1:
        rates = by_arm['taxa_conversao']
        arm_summary.append({
            'tabela': table_id,
            'coluna_arm': arm_col,
            'opcoes_arm': len(by_arm),
            'taxa_min_%': round(float(rates.min() * 100), 2),
            'taxa_max_%': round(float(rates.max() * 100), 2),
            'diferenca_pontos_percentuais': round(float((rates.max() - rates.min()) * 100), 2),
            'desvio_padrao_taxas_%': round(float(rates.std(ddof=0) * 100), 2)
        })

print('\n' + '=' * 90)
print('COMPARAÇÃO: CONVERSÃO MAIS DISTRIBUÍDA ENTRE AS OPÇÕES ARM')
print('=' * 90)
if arm_summary:
    arm_balance_summary = pd.DataFrame(arm_summary).sort_values(
        ['diferenca_pontos_percentuais', 'desvio_padrao_taxas_%']
    )
    print(arm_balance_summary.to_string(index=False))
    print(
        f"\nConversão mais distribuída: {arm_balance_summary.iloc[0]['tabela']} "
        f"(diferença de {arm_balance_summary.iloc[0]['diferenca_pontos_percentuais']:.2f} p.p.)"
    )
else:
    print('Nenhuma tabela com mais de uma opção de arm foi encontrada.')


ANÁLISE DA COLUNA ARM E CONVERSÃO POR OPÇÃO

bank-marketing/bank-additional-full | arm: contact | target: y
           observacoes  conversoes  taxa_conversao_%
contact                                             
cellular         26144        3853             14.74
telephone        15044         787              5.23

bank-marketing-data-set/bank-additional-full | arm: contact | target: y
           observacoes  conversoes  taxa_conversao_%
contact                                             
cellular         26144        3853             14.74
telephone        15044         787              5.23

bank-term-deposit-subscription/bank-full | arm: contact | target: y
           observacoes  conversoes  taxa_conversao_%
contact                                             
cellular         29285        4369             14.92
telephone         2906         390             13.42
unknown          13020         530              4.07

telemarketing-jyb-dataset/test: coluna arm ou target não id